# Pipeline CPU 1 : PyTorch -> ONNX -> OpenVINO Runtime -> CPU
Runtime : CPU (pas de GPU).

In [1]:
!pip -q install openvino onnx onnxscript

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
sys.path.append('/content/drive/MyDrive/tp_deploy')
from common import *
X, y = load_data()
print(X.shape, y[:10])

Mounted at /content/drive
torch.Size([50, 3, 224, 224]) [2 1 5 8 9 3 8 9 0 1]


In [3]:
model = load_model()
dummy = torch.randn(BS, 3, 224, 224)
torch.onnx.export(model, dummy, 'resnet18.onnx', input_names=['input'], output_names=['logits'],
                  opset_version=17, dynamo=False)
import openvino as ov
core = ov.Core()
t = time.perf_counter()
compiled = core.compile_model('resnet18.onnx', 'CPU', {'PERFORMANCE_HINT': 'LATENCY'})
print('compilation OpenVINO :', time.perf_counter() - t, 's')

def predict(b):
    return compiled(b.numpy())[0]

/tmp/ipykernel_1657/869528192.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, 'resnet18.onnx', input_names=['input'], output_names=['logits'],


compilation OpenVINO : 0.2852439190000382 s


In [4]:
res = run_benchmark('cpu_openvino', predict, X, y, device='cpu')

RAPL indisponible : [Errno 2] No such file or directory: '/sys/class/powercap/intel-rapl:0/energy_uj'
total_s      moy=3.5606 std=0.1594 min=3.4111 max=3.7746
throughput   moy=14.0649 std=0.6219 min=13.2466 max=14.6581
batch_ms     moy=712.1055 std=31.8727 min=682.2100 max=754.8852
top1         moy=0.9200 std=0.0000 min=0.9200 max=0.9200
énergie : None
